In [2]:
import os
import traceback
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("❌ HUGGINGFACEHUB_API_TOKEN not found in .env file")
print("✅ Environment setup complete")

✅ Environment setup complete


In [3]:
persist_dir = "../vector_store/complaints_db"
if not os.path.exists(persist_dir):
    persist_dir = "./vector_store/complaints_db"

if not os.path.exists(persist_dir):
    raise RuntimeError(f"❌ Vector database directory not found at {persist_dir}")

try:
    # Device-aware embeddings
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": False}
    )

    vector_db = Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings
    )

    doc_count = len(vector_db.get()["ids"])
    if doc_count == 0:
        raise RuntimeError("❌ Vector DB is empty. Please check your ingestion script.")

    print(f"✅ Vector DB loaded with {doc_count} chunks")
except Exception as e:
    print(f"❌ Failed to load vector database: {e}")
    traceback.print_exc()

Using device: cpu


/var/folders/nf/drxnv59536v7bnknmx3c_1zw0000gn/T/ipykernel_98092/3717823558.py:20: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


✅ Vector DB loaded with 82891 chunks


In [4]:
try:
    llm_endpoint = HuggingFaceEndpoint(
        repo_id="HuggingFaceH4/zephyr-7b-beta",
        task="conversational",
        max_new_tokens=512,
        temperature=0.1,
        huggingfacehub_api_token=HF_TOKEN,
        timeout=120
    )

    llm = ChatHuggingFace(llm=llm_endpoint)
    print("✅ LLM (Zephyr-7B via Chat interface) setup complete")
except Exception as e:
    print(f"❌ LLM Setup failed: {e}")
    traceback.print_exc()

✅ LLM (Zephyr-7B via Chat interface) setup complete


In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question using ONLY the provided context. "
               "If the answer is not in the context, say 'Information not found'."),
    ("user", "Context:\n{context}\n\nQuestion:\n{question}")
])

retriever = vector_db.as_retriever(search_kwargs={"k": 5})  # slightly higher k for more context

def format_docs(docs):
    """Formats retrieved docs and removes duplicate text chunks"""
    if not docs:
        return "No relevant context found."
    
    seen_texts = set()
    unique_chunks = []
    for d in docs:
        if d.page_content not in seen_texts:
            seen_texts.add(d.page_content)
            meta_info = f"[Product: {d.metadata.get('product','N/A')}, " \
                        f"Company: {d.metadata.get('company','N/A')}, " \
                        f"Chunk: {d.metadata.get('chunk_index','N/A')}, " \
                        f"Words: {len(d.page_content.split())}]"
            unique_chunks.append(f"{meta_info}\n{d.page_content}")
    return "\n\n".join(unique_chunks)

In [6]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
print("✅ RAG Chain built successfully")

✅ RAG Chain built successfully


In [7]:
print("\n--- RUNNING SYSTEM TEST ---")
try:
    # Test LLM connectivity
    print("🔍 Testing LLM basic connectivity...")
    test_msg = llm.invoke("Hello, are you ready?")
    print(f"Response: {test_msg.content[:200]}...")  # preview first 200 chars

    # Test full RAG functionality
    question = "What are common complaints about unauthorized credit card charges?"
    print(f"\n🔍 RAG Question: {question}")
    
    response = rag_chain.invoke(question)
    
    print("\n" + "="*50)
    print("🧠 FINAL ANSWER:")
    print("="*50)
    print(response)
    print("="*50)

except Exception:
    print("\n❌ CRITICAL ERROR DURING EXECUTION")
    traceback.print_exc()


--- RUNNING SYSTEM TEST ---
🔍 Testing LLM basic connectivity...
Response: I'm not capable of being ready or not, but I'm always available to assist you with your queries. Please let me know how I can help you today. What would you like me to do for you?...

🔍 RAG Question: What are common complaints about unauthorized credit card charges?

🧠 FINAL ANSWER:
1. Stolen or lost cards: Many people report unauthorized charges on their credit cards after their cards have been lost or stolen, leading to fraudulent transactions. They may not have noticed the loss immediately and only discover the issue when they see charges they did not make.

2. Unfamiliar charges: Some people may notice charges from unfamiliar merchants or companies they do not recognize, indicating that their card information has been compromised.

3. Multiple charges: Multiple charges in quick succession or in large amounts can be a sign of fraud, as it's unlikely that a person would make so many purchases in such a short t